In [12]:


from dataclasses import dataclass
from typing import Tuple
import math


# ===========================================================================
# Q3.61 format constants
# ===========================================================================

WORD_BITS  = 64
FRAC_BITS  = 61
SCALE      = 1 << FRAC_BITS              # 2^61
SIGN_BIT   = 1 << (WORD_BITS - 1)        # 2^63
WORD_MASK  = (1 << WORD_BITS) - 1
Q_MAX_VAL  = (1 << (WORD_BITS - 1)) - 1  # +2^63 - 1
Q_MIN_VAL  = -(1 << (WORD_BITS - 1))     # -2^63


# ===========================================================================
# Conversion helpers: float <-> Q3.61 int <-> hex string
# ===========================================================================

def float_to_q(x: float) -> int:
    """Convert a Python float to a 64-bit signed Q3.61 integer (saturating)."""
    scaled = x * SCALE
    v = int(scaled + 0.5) if scaled >= 0 else int(scaled - 0.5)
    if v > Q_MAX_VAL:
        v = Q_MAX_VAL
    elif v < Q_MIN_VAL:
        v = Q_MIN_VAL
    return v


def q_to_float(v: int) -> float:
    """Convert a signed Q3.61 integer back to float."""
    return v / SCALE


def hex_to_q(h: str) -> int:
    """
    Parse a 64-bit hex string into a signed integer.

    Accepts '0xDEAD...', 'DEAD...', "64'hDEAD...", with optional underscores.
    The hex word is interpreted as a 64-bit two's complement value.
    """
    s = h.strip().lower().replace("_", "")
    if "'h" in s:
        s = s.split("'h", 1)[1]
    if s.startswith("0x"):
        s = s[2:]
    if not s:
        raise ValueError(f"Empty hex string: {h!r}")
    u = int(s, 16) & WORD_MASK
    if u & SIGN_BIT:
        u -= 1 << WORD_BITS
    return u


def q_to_hex(v: int, prefix: str = "0x") -> str:
    """Format a signed Q3.61 integer as a 16-digit (64-bit) hex string."""
    u = v & WORD_MASK
    return f"{prefix}{u:016x}"


def float_to_hex(x: float, prefix: str = "0x") -> str:
    """Convenience: float -> Q3.61 -> hex string."""
    return q_to_hex(float_to_q(x), prefix=prefix)


def hex_to_float(h: str) -> float:
    """Convenience: hex string -> Q3.61 -> float."""
    return q_to_float(hex_to_q(h))


# ===========================================================================
# Q3.61 arithmetic primitives (bit-exact model of the DSP datapath)
# ===========================================================================

def q_mul(a: int, b: int) -> int:
    """
    Multiply two signed Q3.61 integers, returning a signed Q3.61 result.

    Uses Python big-int multiply for an exact 128-bit Q6.122 product, then
    arithmetic right-shift by FRAC_BITS with truncation toward zero (drop
    LSBs). Saturates on overflow.
    """
    full = a * b
    # Truncation toward zero (matches dropping low-order bits in HDL)
    if full >= 0:
        shifted = full >> FRAC_BITS
    else:
        shifted = -((-full) >> FRAC_BITS)
    if shifted > Q_MAX_VAL:
        return Q_MAX_VAL
    if shifted < Q_MIN_VAL:
        return Q_MIN_VAL
    return shifted


def q_add(a: int, b: int) -> int:
    """Add two signed Q3.61 integers with saturation."""
    s = a + b
    if s > Q_MAX_VAL:
        return Q_MAX_VAL
    if s < Q_MIN_VAL:
        return Q_MIN_VAL
    return s


def q_sub(a: int, b: int) -> int:
    """Subtract two signed Q3.61 integers with saturation."""
    return q_add(a, -b)


# ===========================================================================
# Float-domain mixer update (reference)
# ===========================================================================

@dataclass
class Amplitude:
    re: float
    im: float

    def __repr__(self) -> str:
        sign = "+" if self.im >= 0 else "-"
        return f"({self.re:.6f} {sign} {abs(self.im):.6f}j)"


def mixer_pair_update(
    cos_beta: float,
    sin_beta: float,
    p_r: float,
    p_i: float,
    p_br: float,
    p_bi: float,
) -> Tuple[float, float, float, float]:
    """Float-domain reference, returns (p_r_new, p_i_new, p_br_new, p_bi_new)."""
    p_r_new  = cos_beta * p_r  + sin_beta * p_bi
    p_i_new  = cos_beta * p_i  - sin_beta * p_br
    p_br_new = cos_beta * p_br + sin_beta * p_i
    p_bi_new = cos_beta * p_bi - sin_beta * p_r
    return p_r_new, p_i_new, p_br_new, p_bi_new


# ===========================================================================
# Q3.61 fixed-point mixer update (integer arithmetic)
# ===========================================================================

def mixer_pair_update_q(
    cos_beta_q: int,
    sin_beta_q: int,
    p_r_q: int,
    p_i_q: int,
    p_br_q: int,
    p_bi_q: int,
) -> Tuple[int, int, int, int]:
    """
    Bit-exact Q3.61 fixed-point mixer update.

    All inputs and outputs are signed 64-bit Q3.61 integers. This is what
    your SystemVerilog DUT should produce.
    """
    p_r_new  = q_add(q_mul(cos_beta_q, p_r_q),  q_mul(sin_beta_q, p_bi_q))
    p_i_new  = q_sub(q_mul(cos_beta_q, p_i_q),  q_mul(sin_beta_q, p_br_q))
    p_br_new = q_add(q_mul(cos_beta_q, p_br_q), q_mul(sin_beta_q, p_i_q))
    p_bi_new = q_sub(q_mul(cos_beta_q, p_bi_q), q_mul(sin_beta_q, p_r_q))
    return p_r_new, p_i_new, p_br_new, p_bi_new


# ===========================================================================
# Hex-I/O mixer update — the headline helper
# ===========================================================================

def mixer_pair_update_hex(
    cos_beta_hex: str,
    sin_beta_hex: str,
    p_r_hex:  str,
    p_i_hex:  str,
    p_br_hex: str,
    p_bi_hex: str,
) -> Tuple[str, str, str, str]:
    """
    QAOA mixer pair update with Verilog-style hex I/O in Q3.61.

    Each input is a 64-bit signed Q3.61 value, given as a hex string
    (e.g. '0x2000000000000000' = 0.125, "64'h..." style is also accepted).

    Returns (p_r_new_hex, p_i_new_hex, p_br_new_hex, p_bi_new_hex), each as
    a 16-digit, '0x'-prefixed hex string.
    """
    cos_q = hex_to_q(cos_beta_hex)
    sin_q = hex_to_q(sin_beta_hex)
    pr_q  = hex_to_q(p_r_hex)
    pi_q  = hex_to_q(p_i_hex)
    pbr_q = hex_to_q(p_br_hex)
    pbi_q = hex_to_q(p_bi_hex)

    pr_new, pi_new, pbr_new, pbi_new = mixer_pair_update_q(
        cos_q, sin_q, pr_q, pi_q, pbr_q, pbi_q
    )

    return (
        q_to_hex(pr_new),
        q_to_hex(pi_new),
        q_to_hex(pbr_new),
        q_to_hex(pbi_new),
    )


# ===========================================================================
# Self-test
# ===========================================================================

def _matrix_reference(beta: float, p_a_c: complex, p_b_c: complex):
    c, s = math.cos(beta), math.sin(beta)
    return c * p_a_c - 1j * s * p_b_c, -1j * s * p_a_c + c * p_b_c


def _run_tests():
    print("=" * 72)
    print("Q3.61 format constants")
    print("=" * 72)
    print(f"  word bits = {WORD_BITS}, frac bits = {FRAC_BITS}")
    print(f"  range approx [{q_to_float(Q_MIN_VAL):+.6f}, "
          f"{q_to_float(Q_MAX_VAL):+.6f}]")
    print(f"  LSB = 2^-{FRAC_BITS} = {1.0/SCALE:.3e}")
    print(f"  +1.0  -> {float_to_hex(1.0)}")
    print(f"  -1.0  -> {float_to_hex(-1.0)}")
    print(f"  +0.5  -> {float_to_hex(0.5)}")
    print(f"  cos(pi/4) = sin(pi/4) -> {float_to_hex(math.cos(math.pi/4))}")

    print()
    print("=" * 72)
    print("Mixer update — float vs Q3.61 vs matrix reference")
    print("=" * 72)

    cases = [
        (0.0,         1.0 + 0.0j,   0.0 + 0.0j),
        (math.pi / 2, 1.0 + 0.0j,   0.0 + 0.0j),
        (math.pi / 4, 1.0 + 0.0j,   0.0 + 0.0j),
        (math.pi / 3, 0.6 + 0.8j,   0.0 + 0.0j),
        (0.7,         0.3 - 0.4j,   0.5 + 0.2j),
        (-1.2,        0.1 + 0.9j,  -0.4 + 0.3j),
    ]

    print(f"{'beta':>8}  {'|err|_float':>14}  {'|err|_Q3.61':>14}")
    print("-" * 44)
    worst = 0.0
    for beta, pa, pb in cases:
        pa_ref, pb_ref = _matrix_reference(beta, pa, pb)

        pr, pi, pbr, pbi = mixer_pair_update(
            math.cos(beta), math.sin(beta),
            pa.real, pa.imag, pb.real, pb.imag,
        )
        ef = max(abs(complex(pr, pi) - pa_ref),
                 abs(complex(pbr, pbi) - pb_ref))

        out_hex = mixer_pair_update_hex(
            float_to_hex(math.cos(beta)), float_to_hex(math.sin(beta)),
            float_to_hex(pa.real), float_to_hex(pa.imag),
            float_to_hex(pb.real), float_to_hex(pb.imag),
        )
        pa_q = complex(hex_to_float(out_hex[0]), hex_to_float(out_hex[1]))
        pb_q = complex(hex_to_float(out_hex[2]), hex_to_float(out_hex[3]))
        eq = max(abs(pa_q - pa_ref), abs(pb_q - pb_ref))
        worst = max(worst, eq)

        print(f"{beta:>8.4f}  {ef:>14.2e}  {eq:>14.2e}")

    print(f"\nWorst-case Q3.61 error: {worst:.2e}  "
          f"(LSB = {1.0/SCALE:.2e})")

    print()
    print("=" * 72)
    print("Example hex-I/O call")
    print("=" * 72)
    beta = math.pi / 3
    cos_h = float_to_hex(math.cos(beta))
    sin_h = float_to_hex(math.sin(beta))
    pr_h, pi_h  = float_to_hex(0.6), float_to_hex(0.8)
    pbr_h, pbi_h = float_to_hex(0.0), float_to_hex(0.0)

    print("Inputs (Q3.61 hex):")
    for name, h in [("cos_beta", cos_h), ("sin_beta", sin_h),
                    ("p_r", pr_h), ("p_i", pi_h),
                    ("p_br", pbr_h), ("p_bi", pbi_h)]:
        print(f"  {name:>8} = {h}  (= {hex_to_float(h):+.10f})")

    out = mixer_pair_update_hex(cos_h, sin_h, pr_h, pi_h, pbr_h, pbi_h)
    print("\nOutputs (Q3.61 hex):")
    for name, h in zip(("p_r_new", "p_i_new", "p_br_new", "p_bi_new"), out):
        print(f"  {name:>8} = {h}  (= {hex_to_float(h):+.10f})")


if __name__ == "__main__":
    _run_tests()

Q3.61 format constants
  word bits = 64, frac bits = 61
  range approx [-4.000000, +4.000000]
  LSB = 2^-61 = 4.337e-19
  +1.0  -> 0x2000000000000000
  -1.0  -> 0xe000000000000000
  +0.5  -> 0x1000000000000000
  cos(pi/4) = sin(pi/4) -> 0x16a09e667f3bcd00

Mixer update — float vs Q3.61 vs matrix reference
    beta     |err|_float     |err|_Q3.61
--------------------------------------------
  0.0000        0.00e+00        0.00e+00
  1.5708        0.00e+00        8.33e-20
  0.7854        0.00e+00        0.00e+00
  1.0472        0.00e+00        0.00e+00
  0.7000        0.00e+00        5.55e-17
 -1.2000        0.00e+00        1.11e-16

Worst-case Q3.61 error: 1.11e-16  (LSB = 4.34e-19)

Example hex-I/O call
Inputs (Q3.61 hex):
  cos_beta = 0x1000000000000100  (= +0.5000000000)
  sin_beta = 0x1bb67ae8584caa00  (= +0.8660254038)
       p_r = 0x1333333333333300  (= +0.6000000000)
       p_i = 0x1999999999999a00  (= +0.8000000000)
      p_br = 0x0000000000000000  (= +0.0000000000)
      p_bi =

In [ ]:
# Method A: Build a named-constants table (recommended for repeated use)
# ---------------------------------------------------------------------------

Q = {
    # Common amplitudes
    "ZERO":    float_to_hex(0.0),
    "ONE":     float_to_hex(1.0),
    "NEG_ONE": float_to_hex(-1.0),
    "HALF":    float_to_hex(0.5),

    # Trig values at standard mixer angles
    "COS_PI_8":  float_to_hex(math.cos(math.pi / 8)),
    "SIN_PI_8":  float_to_hex(math.sin(math.pi / 8)),
    "COS_PI_4":  float_to_hex(math.cos(math.pi / 4)),
    "SIN_PI_4":  float_to_hex(math.sin(math.pi / 4)),
    "COS_PI_3":  float_to_hex(math.cos(math.pi / 3)),
    "SIN_PI_3":  float_to_hex(math.sin(math.pi / 3)),

    # Equal-superposition amplitudes (e.g. 1/sqrt(2) and 1/2 for 2-qubit |+>)
    "INV_SQRT2": float_to_hex(1.0 / math.sqrt(2)),
    "INV_2":     float_to_hex(0.5),
}

print("Named hex constants:")
for k, v in Q.items():
    print(f"  {k:>10} = {v}   (= {hex_to_float(v):+.10f})")


# ---------------------------------------------------------------------------
# Method B: Pass new hex values directly into the mixer call
# ---------------------------------------------------------------------------

print("\nExample: mixer at beta = pi/8, p_a = 1/sqrt(2)*(1+i), p_b = 0")

out = mixer_pair_update_hex(
    cos_beta_hex = Q["64'h19518bebead3c500"],
    sin_beta_hex = Q["64'h1391d5072ee48700"],
    p_r_hex      = Q["64'h0c9d0561b6b10a00"],
    p_i_hex      = Q["64'he2973c6b0f92a900"],
    p_br_hex     = Q["64'h0a588d4eeeb1ca00"],
    p_bi_hex     = Q["64'he1b7f7359e1b3400"],
)

for name, h in zip(("p_r_new", "p_i_new", "p_br_new", "p_bi_new"), out):
    print(f"  {name:>8} = {h}  (= {hex_to_float(h):+.10f})")


# ---------------------------------------------------------------------------
# Method C: Sweep a parameter — generate hex values on the fly
# ---------------------------------------------------------------------------

print("\nSweep: cos(beta) for beta = 0, pi/8, pi/4, 3pi/8, pi/2")
for beta in [0, math.pi/8, math.pi/4, 3*math.pi/8, math.pi/2]:
    cos_h = float_to_hex(math.cos(beta))
    sin_h = float_to_hex(math.sin(beta))
    print(f"  beta = {beta:.4f}  cos = {cos_h}   sin = {sin_h}")


# ---------------------------------------------------------------------------
# Method D: Hand-craft a hex value when you want bit-exact control
# ---------------------------------------------------------------------------

# In Q3.61, 1.0 has binary form '01.00...00' = 0x2000000000000000
# Half of that is 0x1000000000000000 = 0.5
# A quarter is   0x0800000000000000 = 0.25
# Sign-flip means two's complement: -0.25 = 0xf800000000000000

print("\nHand-crafted values:")
for h in ["64'h19518bebead3c500",
          "64'h1391d5072ee48700",
          "64'he2973c6b0f92a900",
          "64'h0a588d4eeeb1ca00",
          "64'he1b7f7359e1b3400"]:   # = 2.0, near the top of Q3.61 range
    print(f"  {h}  = {hex_to_float(h):+.10f}")

Named hex constants:
        ZERO = 0x0000000000000000   (= +0.0000000000)
         ONE = 0x2000000000000000   (= +1.0000000000)
     NEG_ONE = 0xe000000000000000   (= -1.0000000000)
        HALF = 0x1000000000000000   (= +0.5000000000)
    COS_PI_8 = 0x1d906bcf328d4600   (= +0.9238795325)
    SIN_PI_8 = 0x0c3ef1535754b180   (= +0.3826834324)
    COS_PI_4 = 0x16a09e667f3bcd00   (= +0.7071067812)
    SIN_PI_4 = 0x16a09e667f3bcd00   (= +0.7071067812)
    COS_PI_3 = 0x1000000000000100   (= +0.5000000000)
    SIN_PI_3 = 0x1bb67ae8584caa00   (= +0.8660254038)
   INV_SQRT2 = 0x16a09e667f3bcc00   (= +0.7071067812)
       INV_2 = 0x1000000000000000   (= +0.5000000000)

Example: mixer at beta = pi/8, p_a = 1/sqrt(2)*(1+i), p_b = 0


KeyError: "64'h19518bebead3c500"

In [6]:
 
# Echo inputs (with float decode for sanity)
print("Inputs:")
for name, h in [
    ("cos_beta", cos_beta_hex), ("sin_beta", sin_beta_hex),
    ("p_r",      p_r_hex),      ("p_i",      p_i_hex),
    ("p_br",     p_br_hex),     ("p_bi",     p_bi_hex),
]:
    print(f"  {name:>8} = {h}   (= {hex_to_float(h):+.10f})")
 
# Compute
p_r_new, p_i_new, p_br_new, p_bi_new = mixer_pair_update_hex(
    cos_beta_hex, sin_beta_hex,
    p_r_hex, p_i_hex,
    p_br_hex, p_bi_hex,
)
 
print("\nOutputs:")
for name, h in [
    ("p_r_new",  p_r_new),  ("p_i_new",  p_i_new),
    ("p_br_new", p_br_new), ("p_bi_new", p_bi_new),
]:
    print(f"  {name:>8} = {h}   (= {hex_to_float(h):+.10f})")
 

Inputs:


NameError: name 'cos_beta_hex' is not defined